In [44]:
import math
import random
import json

In [45]:
def extended_gcd(a, b):
    sa = 1 if a >= 0 else -1
    sb = 1 if b >= 0 else -1
    a, b = abs(a), abs(b)
    
    if a == 0:
        return b, 0, sb
    
    g, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    
    return g, x * sa, y * sb

def mod_inverse(a, m):
    a %= m 
    g, x, _ = extended_gcd(a, m)
    
    return x % m if g==1 else None

def pow_mod(base, exp, mod):
    result = 1
    base %= mod
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % mod
        exp //= 2
        base = (base * base) % mod
    return result

In [46]:
def check_math_menu():
    print("\n=== ПРОВЕРКА МАТЕМАТИКИ ===")
    choice = input("Выберите действие: 1. Проверка алгоритма Евклида 2. Проверка возведения в степень 0. Выход").strip()
    if choice == '1':
        try:
            a = int(input("Введите число a: "))
            b = int(input("Введите число b: "))
            
            g, x, y = extended_gcd(a, b)

            print(f"Вход: a={a}, b={b}")
            print(f"НОД({a}, {b})={g}")
            print(f"Коэффициенты Безу: x={x}, y={y}")
            print(f"Проверка тождества Безу: {a}*({x}) + {b}*({y}) = {a*x + b*y}")
                        
        except ValueError:
            print("Ошибка ввода: необходимо ввести целые числа.")
            
    elif choice == '2':
        try:
            base = int(input("Введите основание (base): "))
            exp = int(input("Введите показатель степени (exp): "))
            mod = int(input("Введите модуль (mod): "))
            
            if mod <= 0:
                print("Ошибка: модуль должен быть положительным числом.")
                return
                
            result = pow_mod(base, exp, mod)
            
            print(f"{base}^{exp} mod {mod} = {result}")
                
        except ValueError:
            print("Ошибка ввода: необходимо ввести целые числа.")
            
    elif choice == '0':
        print("Выход из проверки математики.")
        return False
        
    else:
        print("Неверный выбор. Попробуйте снова.")
        
    return True

In [47]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def generate_random_e(phi_n):
    while True:
        e = random.randrange(3, phi_n, 2)
        if math.gcd(e, phi_n) == 1:
            return e

def generate_rsa_keypair(p, q, e):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError("p и q должны быть простыми")
    
    n = p * q
    phi_n = (p - 1) * (q - 1)
    
    if math.gcd(e, phi_n) != 1:
        raise ValueError(f"e={e} не взаимно просто с φ(n)")
        
    d = mod_inverse(e, phi_n)
    if d is None:
        raise ValueError("Не удалось найти обратный элемент d")
        
    return {
        'n': n, 
        'e': e, 
        'd': d
    }


In [48]:
def generate_rsa_keypair(p, q, e):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError("p и q должны быть простыми")
    
    n = p * q
    phi_n = (p - 1) * (q - 1)
    
    if math.gcd(e, phi_n) != 1:
        raise ValueError(f"e={e} не взаимно просто с φ(n)")
        
    d = mod_inverse(e, phi_n)
    if d is None:
        raise ValueError("Не удалось найти обратный элемент d")
        
    return {
        'n': n, 
        'e': e, 
        'd': d
    }

def generate_keys_menu():
    print("\n=== ГЕНЕРАЦИЯ КЛЮЧЕЙ RSA ===")
    
    p_input = input("Введите простое число p [по умолч.: 227]: ").strip()
    p = int(p_input) if p_input else 227
    
    q_input = input("Введите простое число q [по умолч.: 373]: ").strip()
    q = int(q_input) if q_input else 373
    
    keys = []
    try:
        for i in range(1, 4):
            print(f"\n--- Генерация пары ключей #{i} ---")
            
            e_str = input(f"Введите открытую экспоненту e для пары #{i} [Enter = случайно]: ").strip()
            
            if not e_str:
                phi_n_temp = (p - 1) * (q - 1)
                e = generate_random_e(phi_n_temp)
                print(f"Случайно сгенерировано e = {e}")
            else:
                e = int(e_str)
            
            kp = generate_rsa_keypair(p, q, e)
            keys.append(kp)
            
            print(f"Пара #{i}: e={kp['e']}, d={kp['d']}, n={kp['n']}")
            
        filename = "rsa_keys.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump({
                'params': {'p': p, 'q': q},
                'keys': keys
            }, f, indent=2, ensure_ascii=False)
                
        print(f"\nКлючи сохранены в '{filename}'")
        
    except ValueError as ve:
        print(f"Ошибка генерации: {ve}")
    except Exception as ex:
        print(f"Непредвиденная ошибка: {ex}")

In [49]:
ALPHABET_RU = "АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
CHAR_TO_CODE = {char: str(10 + i) for i, char in enumerate(ALPHABET_RU)}
CHAR_TO_CODE[' '] = '99'

def prepare_text(text):
    """Подготовка открытого текста: верхний регистр, только русские буквы и пробелы"""
    cleaned = ""
    for char in text.upper():
        if char in ALPHABET_RU or char == ' ':
            cleaned += char
    return cleaned

def text_to_numeric_string(text):
    """Преобразование текста в строку цифровых кодов"""
    numeric_parts = []
    for char in text:
        code = CHAR_TO_CODE.get(char)
        if code is not None:
            numeric_parts.append(code)
    return "".join(numeric_parts)

In [50]:
def split_into_blocks(numeric_str, n):
    blocks = []
    max_len = len(str(n))
    i = 0
    while i < len(numeric_str):
        block_len = min(max_len, len(numeric_str) - i)
        while block_len > 1:
            candidate = numeric_str[i:i + block_len]
            next_idx = i + block_len
            if int(candidate) < n and (next_idx >= len(numeric_str) or numeric_str[next_idx] != '0'):
                break
            block_len -= 1
        blocks.append(int(numeric_str[i:i + block_len]))
        i += block_len
    return blocks

In [51]:
def encrypt_menu():
    print("\n=== ШИФРОВАНИЕ RSA ===")
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    e = n = None
    if keys:
        print("Доступные ключи:")
        for i, k in enumerate(keys, 1):
            print(f"  {i}. e={k['e']}, n={k['n']}")
        choice = input("Номер ключа [Enter = ввести свой]: ").strip()
        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(keys):
                    e, n = keys[idx]['e'], keys[idx]['n']
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл ключей не найден — введите ключ вручную.")

    if e is None:
        try:
            e = int(input("Введите e: "))
            n = int(input("Введите n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Ключ: e={e}, n={n}")
    text = prepare_text(input("Открытый текст: "))
    if not text:
        print("Текст пуст.")
        return

    blocks = split_into_blocks(text_to_numeric_string(text), n)
    print(f"Блоки открытого текста: {blocks}")

    block_len = len(str(n))
    cipher = [pow_mod(m, e, n) for m in blocks]
    print(f"ШИФР-ТЕКСТ: {''.join(str(c).zfill(block_len) for c in cipher)}")

In [52]:
CODE_TO_CHAR = {v: k for k, v in CHAR_TO_CODE.items()}

def decrypt_blocks_to_text(decrypted_blocks):
    numeric_str = "".join(str(b) for b in decrypted_blocks)

    text = ""
    for i in range(0, len(numeric_str), 2):
        code_pair = numeric_str[i:i+2]
        char = CODE_TO_CHAR.get(code_pair, '?')
        text += char
        
    return text

In [53]:
def decrypt_menu():
    print("\n=== РАСШИФРОВАНИЕ RSA ===")
    try:
        with open("rsa_keys.json", encoding="utf-8") as f:
            keys = json.load(f)["keys"]
    except (FileNotFoundError, KeyError, ValueError):
        keys = []

    d = n = None
    if keys:
        print("Доступные ключи:")
        for i, k in enumerate(keys, 1):
            print(f"  {i}. d={k['d']}, n={k['n']}")
        choice = input("Номер ключа [Enter = ввести свой]: ").strip()
        if choice:
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(keys):
                    d, n = keys[idx]['d'], keys[idx]['n']
                else:
                    print("Неверный номер ключа.")
                    return
            except ValueError:
                print("Ошибка ввода номера.")
                return
    else:
        print("Файл ключей не найден — введите ключ вручную.")

    if d is None:
        try:
            d = int(input("Введите d: "))
            n = int(input("Введите n: "))
        except ValueError:
            print("Ошибка ввода: нужны целые числа.")
            return

    print(f"Ключ: d={d}, n={n}")
    s = input("Шифр-текст (одной строкой цифр): ").strip()
    block_len = len(str(n))
    if not s.isdigit() or len(s) % block_len != 0:
        print(f"Ошибка: строка должна состоять из цифр, длина кратна {block_len}.")
        return

    cipher = [int(s[i:i + block_len]) for i in range(0, len(s), block_len)]
    blocks = [pow_mod(c, d, n) for c in cipher]
    print(f"Расшифрованные блоки: {blocks}")
    print(f"ОТКРЫТЫЙ ТЕКСТ: {decrypt_blocks_to_text(blocks)}")

In [54]:
def crypto_menu():
    while True:
        print("\n=== ШИФРОВАНИЕ / РАСШИФРОВАНИЕ ===")
        choice = input("1. Зашифровать 2. Расшифровать -> ").strip()

        if choice == '1':
            encrypt_menu()
        elif choice == '2':
            decrypt_menu()
        elif choice == '0':
            break
        else:
            print("Неверный выбор, попробуйте снова.")


def main_menu():
    while True:
        print("\n=== ГЛАВНОЕ МЕНЮ (RSA) ===")
        choice = input("1. Проверка математики 2. Генерация ключей 3. Шифрование/расшифрование -> ").strip()

        if choice == '1':
            while check_math_menu():
                pass
        elif choice == '2':
            generate_keys_menu()
        elif choice == '3':
            crypto_menu()
        elif choice == '0':
            print("Выход из программы.")
            break
        else:
            print("Неверный выбор, попробуйте снова.")

main_menu()


=== ГЛАВНОЕ МЕНЮ (RSA) ===

=== ШИФРОВАНИЕ / РАСШИФРОВАНИЕ ===

=== РАСШИФРОВАНИЕ RSA ===
Доступные ключи:
  1. d=65899, n=84671
  2. d=52213, n=84671
  3. d=56807, n=84671
Ключ: d=65899, n=84671
Ошибка: строка должна состоять из цифр, длина кратна 5.

=== ШИФРОВАНИЕ / РАСШИФРОВАНИЕ ===
Неверный выбор, попробуйте снова.

=== ШИФРОВАНИЕ / РАСШИФРОВАНИЕ ===

=== ГЛАВНОЕ МЕНЮ (RSA) ===
Выход из программы.
